# 02 ตรวจคุณภาพและสำรวจข้อมูล
ต่อจาก notebook 01 สมาชิกจะเขียนการวิเคราะห์และกราฟเอง ตัวช่วยมีเพียงการอ่านไฟล์และรายการตรวจพื้นฐาน

ก่อนใช้ข้อมูลตัดสินใจเรื่อง feature/preprocessing ให้กำหนด test ที่กันไว้กับ notebook 03 ใช้ train สำรวจรูปแบบเพื่อไม่ให้ test มีผลต่อการออกแบบ


## เจ้าของงานและการส่งต่อสำหรับทีม 5 คน
เจ้าของช่วงนี้ M1 ผู้ตรวจ M4: รับ split จาก M2 ก่อนทำ EDA เชิงตัดสินใจบน train ส่งกราฟและ quality report ให้ M2 ใช้ออกแบบ preprocessing

M1–M5 เป็นรหัสบทบาทเสนอ ให้กลุ่มใส่ชื่อจริงและตกลงผู้ตรวจ ไม่ใช่หลักฐานว่างานทำเสร็จแล้ว ทุกคนบันทึก contribution/prompts และเขียนรายงานส่วนของตน


## เตรียมงานใน Colab
บันทึกสำเนา notebook ใน Drive ก่อน รัน cell นี้เพื่อเตรียม repository ใน runtime เมื่อมีโฟลเดอร์เดิมจะใช้ต่อโดยไม่เขียนทับ สำรองไฟล์ใน Files แยกจาก notebook ก่อนปิด runtime

ใช้ CPU runtime ได้ ไม่ต้องติดตั้งโปรแกรมในเครื่อง


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/Bicycle_Rental_Prediction')]
ROOT = next((p for p in candidates if (p / 'config/project.json').is_file()), None)
if ROOT is None:
    if not Path('/content').is_dir():
        raise RuntimeError('เปิด notebook นี้บน Google Colab และเชื่อม runtime ก่อน')
    ROOT = Path('/content/Bicycle_Rental_Prediction')
    if ROOT.exists():
        raise FileExistsError('พบโฟลเดอร์เดิมที่ยังอ่าน config ไม่ได้ ตรวจและสำรองงานก่อนแก้')
    subprocess.run(['git', 'clone',
                    'https://github.com/Supot77/Bicycle_Rental_Prediction.git',
                    str(ROOT)], check=True)

# Keep Colab's existing scientific stack; install only packages that are missing.
packages = {'pandas': 'pandas', 'sklearn': 'scikit-learn',
            'matplotlib': 'matplotlib', 'joblib': 'joblib'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from starter_support import load_config, configuration_problems, resolve_dataset_path
config = load_config(ROOT)
problems = configuration_problems(config)
if problems:
    raise ValueError('\n'.join(problems))
print('พร้อมทำงานบน Colab: ใช้โฟลเดอร์ repository เดิมถ้ามี โดยไม่ pull/reset ทับงาน')
print('ค่า null ใน config เป็นสถานะรอยืนยัน ไม่ใช่ข้อมูลที่ต้องเดา')


In [ ]:
df = None
path = resolve_dataset_path(config, ROOT)
if path is None:
    print('ยังไม่ได้ตั้ง dataset.path: อ่านโจทย์และคำถามก่อนได้ ยังไม่มีข้อมูลให้วิเคราะห์')
elif not path.is_file():
    print('ยังไม่พบไฟล์: วาง dataset จากอาจารย์ใน data/raw ตาม config')
else:
    import pandas as pd
    df = pd.read_csv(path, encoding=config['dataset']['encoding'],
                     sep=config['dataset']['delimiter'])
    print('อ่านไฟล์ได้แล้ว ต้องตรวจความหมายและคุณภาพก่อนใช้งาน')


In [ ]:
if df is not None:
    print('จำนวนแถว/คอลัมน์:', df.shape)
    print('ค่าหายต่อคอลัมน์:')
    print(df.isna().sum())
    print('แถวที่ซ้ำทั้งแถว:', df.duplicated().sum())
else:
    print('ยังไม่วิเคราะห์ข้อมูลจนกว่าจะได้รับไฟล์จริง')


## งานตรวจคุณภาพ
ตรวจ dtype หน่วย missing แถวซ้ำ ความหมายค่าศูนย์ ค่าที่เป็นไปไม่ได้ และเวลาที่ขาด แถวซ้ำทั้งแถวเป็นเพียงจุดเริ่มต้น อาจต้องตรวจ key เช่น เวลาและสถานีเพิ่มเติม

อย่าเติมค่าหรือทิ้งข้อมูลใน cell นี้ทันที บันทึกสิ่งที่พบ เหตุผลที่สงสัย และวิธีที่เสนอให้กลุ่มคุยก่อน


## งานสำรวจที่สมาชิกเลือกเองตามคอลัมน์จริง
สร้างกราฟที่ตอบคำถามอย่างน้อยหลายประเด็น เช่น target กระจายอย่างไร ต่างกันตามชั่วโมง/วันหรือไม่ และสัมพันธ์กับอากาศอย่างไร ใช้เฉพาะตัวแปรที่มีจริง

ใต้แต่ละกราฟเขียน: เห็นอะไร → อาจมีผลต่อการฝึก/การใช้งานอย่างไร → ยังสรุปอะไรไม่ได้ ความสัมพันธ์ไม่ใช่หลักฐานเหตุและผล


In [ ]:
# เขียน code เลือก train และสร้างกราฟของกลุ่มเอง
# ใช้ matplotlib/pandas ได้ บันทึกว่ารูปนี้ใช้ subset ใด


## สิ่งส่งต่อและจุดตรวจ
สรุปปัญหาข้อมูล กราฟพร้อมคำอธิบาย และคำถามค้าง ผู้ตรวจต้องอธิบายได้ว่าทำไม outlier บางรายการไม่ควรถูกลบทิ้งเพียงเพราะสูง และค่าศูนย์หมายถึงอะไรใน dataset นี้


## ก่อนส่งต่อ
เก็บข้อสังเกตและหลักฐานที่เผยแพร่ได้ไว้ใน evidence/results ให้เพื่อนอีกคนตรวจและลองทำซ้ำ จากนั้นล้าง outputs และ execution counts ก่อน commit notebook อย่าใส่ผลสมมติเป็นผลทดลองจริง
